# 2) A NN with more hidden layers and comparison of optimization methods

In [27]:
import torch
import torch.nn as nn

from collections.abc import Sequence           # to connect hidden layers sequentially

import torch.optim as optim                    # imports PyTorch optimization tools, such as Adam

In [38]:
# IMPORT DATASET MNIST

from sklearn.model_selection import train_test_split

from torchvision.datasets import MNIST


data = MNIST('mnist', download=True)

images, labels = data.data, data.targets     #{x_i, y_i}_i : dataset

X = images.view(-1, 28*28).float() / 255     #transforms images into model-ready input


#SPLIT DATASET INTO TRAINING + VALIDATION (unseen data for tuning the model)
X_train, X_valid, y_train, y_valid = train_test_split(X, labels, test_size=0.1, random_state=0)          #10% for validation


We decide for our NN, to put **ReLu** as activation function only between fully connected hidden layers.

The advantage of ReLu is the fact that it is a continuous function and its derivatives are 0 or 1, which speeds the backward phase where the gradient is computed.

## NN with more hidden layers, explicit version

In [28]:
class TorchMLP_explicit(nn.Module):

  #3 LINEAR LAYERS + 1 ACTIVATION FUNCTION ReLu
  def __init__(self, input_dim, hidden_dim_1, hidden_dim_2, hidden_dim_3, output_dim):

    super().__init__()

    self.linear1 = nn.Linear(input_dim, hidden_dim_1)
    self.linear2 = nn.Linear(hidden_dim_1, hidden_dim_2)
    self.linear3 = nn.Linear(hidden_dim_2, hidden_dim_3)
    self.linear4 = nn.Linear(hidden_dim_3, output_dim)
    self.relu = nn.ReLU()

  #DEFINE FORWARD PHASE
  def forward(self, x):
    x = self.linear1(x)
    x = self.linear2(x)
    x = self.linear3(x)
    x = self.relu(x)
    return self.linear4(x)



In [29]:

model_exp = TorchMLP_explicit(784, 100, 200, 100, 10)

## NN with more hidden layers, implicit version

Fast method to define hidden layers

In [30]:

class TorchMLP(nn.Module):
    
    def __init__(self, input_dim: int, hidden_dims: Sequence[int], output_dim: int):
        
        super().__init__()

        dims = [input_dim, *hidden_dims, output_dim]                   # list of layers dimensions / number of units

        layers = []                                                    # contains fully connections of layers / arrows

        for i, (in_dim, out_dim) in enumerate(zip(dims, dims[1:])):    # enumerate: run on couples
                                                                       # zip : pairs each couple
                                                                       # [1: ] : takes off element 0
                                                                       # connect adjacent layers
            
            layers.append(nn.Linear(in_dim, out_dim))                  # linearly

            layers.append(nn.ReLU())                                   # add ReLu after each hidden layer

            layers = layers[:-1]                                       # remove ReLu applied to output


        self.model = nn.Sequential(*layers)                            # run layers sequentially

    def forward(self, x):                                              # forward computations sequential
        
        return self.model(x)


In [31]:
#CALL THE MODEL

hidden_layers_dim = [100, 200, 100]

model = TorchMLP(784, hidden_layers_dim, 10)

# LOSS

loss = nn.CrossEntropyLoss()


## Comparison of optimization methods

We consider the NN model_lr, with cross entropy loss.

**OPTIMIZATION PHASE:** an optimization algorithm minimizes the loss function $L$ over the weights and updates the weights as:
$$
W_{i} \leftarrow W_{i} - \nu \cdot \nabla_{W_{i}}(L),
$$
where $\nu$ is the learning rate.

The gradient is subtracted from the weights with a step of $\nu$ at each iteration of the algorithm.

### Gradient Descent

> Classic version, where $\nu > 0$ is constant (and small).

### Mini-batch Stochastic Gradient Descent (Mini-batch SDG)

> The dataset is divided into small random subsets called mini-batches, all of the same size. The optimization phase is performed on each of them at a time. This method increases the randomness, and avoid that the loss gets stucked into false minima or flat regions.

### Adam

> Adaptative method, where the learning rate is not constant and adapts to the gradient size at every iteration. If the gradient is large, the learning rate decreases. If the gradient is small, the learning rate increases. Thanks to this, Adam avoids that during the minimization process, the loss gets stucked in false minima or flat regions.

In [34]:
#OPTIMIZER Adam

optimizer_adam = optim.Adam(model.parameters())


In [ ]:
# TRAINING

for epoch in range(1000):     

  optimizer.zero_grad()
  y_pred = model(X_train)              
  loss_result = loss(y_pred, torch_y_train)       #LOSS COMPUTATION: L(y_pred, y_train)
  loss_result.backward()                          #BACKWARD PHASE: gradient of loss over the weights is computed: dL(w)/dw
  optimizer.step()                                #OPTIMIZATION PHASE: update weights: w <- w + (learning_rate)*(gradient)


  if epoch % 10 == 0:                                     #EVALUATION PHASE: every 10 epochs
    with torch.no_grad():
      pred_valid = torch_mlp(torch_X_valid.to('cuda'))

    _, prediction = torch.ax(pred_valid, dim=1)
    print(accuracy_score(y_valid, prediction.to('cpu')))


NameError: name 'torch_mlp' is not defined

In [ ]:
for param in model.parameters():
  print(param.numel())

print('Total : ', sum(param.numel() for param in model.parameters() ))
